# 07  -  Evaluation & Model Comparison

**Goal:** Compute all classification metrics, draw confusion matrices, compare every model side-by-side, and select the best one by MCC  -  exactly as `final_model_comparison.py` does after a full experiment run.

**Modules used:** `src/evaluation/metrics.py`, `src/evaluation/reports.py`, `src/evaluation/final_model_comparison.py`

---

## 0  -  Imports

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.evaluation.metrics import compute_classification_metrics, summarize_fold_metrics
from src.utils.seed import set_global_seed

set_global_seed(42)

---
## 1  -  Understanding `compute_classification_metrics`

This function is the single source of truth for all evaluation. It accepts:
- `y_true`  -  ground-truth binary labels (0 = Bear, 1 = Bull)
- `y_pred`  -  predicted binary labels
- `y_prob`  -  predicted probabilities (optional, enables ROC-AUC)

And returns a dict with all metrics.

In [ ]:
# Synthetic example  -  replace with real predictions once the pipeline has been run
rng = np.random.default_rng(42)
n   = 300
y_true = rng.integers(0, 2, size=n)

# Simulate a reasonably good model
noise    = rng.uniform(-0.3, 0.3, size=n)
y_prob   = np.clip(y_true.astype(float) + noise, 0.0, 1.0)
y_pred   = (y_prob >= 0.50).astype(int)

metrics = compute_classification_metrics(y_true, y_pred, y_prob)

for k, v in metrics.items():
    if k != 'confusion_matrix':
        print(f'{k:<25} : {v}')

---
## 2  -  Why MCC?

Matthews Correlation Coefficient is defined as:

$$\text{MCC} = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

- **Range:** [-1, 1]  -  0 means random, 1 means perfect, -1 means perfectly wrong.
- **Advantage over F1:** Considers all four quadrants of the confusion matrix. A model that always predicts "Bull" on an imbalanced dataset scores high F1 but MCC ~ 0.
- **Advantage over Accuracy:** Not inflated by class imbalance.

In [ ]:
# Demonstrate the "always predict Bull" failure mode
y_always_bull = np.ones_like(y_true)
m_always_bull = compute_classification_metrics(y_true, y_always_bull)

print('--- Always predict Bull ---')
print(f'Accuracy         : {m_always_bull["accuracy"]:.4f}')
print(f'F1               : {m_always_bull["f1"]:.4f}')
print(f'MCC              : {m_always_bull["mcc"]:.4f}  <- correctly penalised')
print(f'Balanced Accuracy: {m_always_bull["balanced_accuracy"]:.4f}')

---
## 3  -  Confusion matrix visualisation

In [ ]:
def plot_confusion_matrix(cm, title='Confusion Matrix', ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 3))
    labels = ['Bear (0)', 'Bull (1)']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels,
                linewidths=0.5, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title, fontsize=11)

cm = np.array(metrics['confusion_matrix'])
fig, ax = plt.subplots(figsize=(5, 4))
plot_confusion_matrix(cm, 'Example Model  -  Confusion Matrix', ax=ax)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TP={tp}  FP={fp}  FN={fn}  TN={tn}')
print(f'Precision = TP/(TP+FP) = {tp/(tp+fp):.3f}')
print(f'Recall    = TP/(TP+FN) = {tp/(tp+fn):.3f}')

---
## 4  -  Cross-fold aggregation with `summarize_fold_metrics`

In [ ]:
# Synthetic example for fold aggregation with probability metrics.
simulated_folds = []
for _ in range(4):
    y_true_fold = rng.integers(0, 2, 200)
    y_prob_fold = np.clip(y_true_fold.astype(float) * 0.55 + rng.uniform(0.0, 0.55, 200), 0, 1)
    y_pred_fold = (y_prob_fold >= 0.50).astype(int)
    simulated_folds.append(compute_classification_metrics(y_true_fold, y_pred_fold, y_prob_fold))

summary = summarize_fold_metrics(simulated_folds)

key_metrics = ['mcc', 'f1', 'balanced_accuracy', 'roc_auc', 'pr_auc']
print(f'{"Metric":<20}  {"Mean":>8}  {"Std":>8}')
print('-' * 40)
for k in key_metrics:
    mean = summary.get(f'{k}_mean') or 0
    std = summary.get(f'{k}_std') or 0
    print(f'{k:<20}  {mean:>8.4f}  {std:>8.4f}')


## 5 - Multi-model comparison

This section reads the current `run_experiment.py` report schema when artifacts exist. If no report has been generated yet, it falls back to a small illustrative table so the plots remain runnable.


In [ ]:
import json

metrics_dir = ROOT / 'artifacts' / 'metrics'
report_files = sorted(metrics_dir.glob('*_report.json'))
rows = []

for report_file in report_files:
    with open(report_file, encoding='utf-8') as f:
        report = json.load(f)
    for candidate in report.get('cv_candidates', []):
        cv = candidate.get('cv_summary', {})
        rows.append({
            'experiment': report.get('experiment_name'),
            'model': candidate.get('model_name'),
            'variant': candidate.get('variant_tag'),
            'scaler': candidate.get('scaler_name'),
            'mcc_mean': cv.get('mcc_mean'),
            'mcc_std': cv.get('mcc_std'),
            'f1_mean': cv.get('f1_mean'),
            'bal_acc_mean': cv.get('balanced_accuracy_mean'),
            'pr_auc_mean': cv.get('pr_auc_mean'),
        })

if rows:
    cmp_df = pd.DataFrame(rows).sort_values('mcc_mean', ascending=False).reset_index(drop=True)
else:
    fallback_rows = [
        {'model': 'logreg', 'variant': 'q40_lb21', 'scaler': 'standard', 'mcc_mean': 0.08, 'mcc_std': 0.04, 'f1_mean': 0.54, 'bal_acc_mean': 0.54, 'pr_auc_mean': 0.55},
        {'model': 'random_forest', 'variant': 'q40_lb21', 'scaler': 'standard', 'mcc_mean': 0.10, 'mcc_std': 0.05, 'f1_mean': 0.55, 'bal_acc_mean': 0.55, 'pr_auc_mean': 0.56},
        {'model': 'gru', 'variant': 'q40_lb21', 'scaler': 'standard', 'mcc_mean': 0.09, 'mcc_std': 0.06, 'f1_mean': 0.54, 'bal_acc_mean': 0.54, 'pr_auc_mean': 0.55},
    ]
    cmp_df = pd.DataFrame(fallback_rows)
    print('No experiment report found yet. Showing illustrative fallback rows.')

cmp_df.head(15)


### MCC comparison chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['gold' if i == 0 else 'steelblue' for i in range(len(cmp_df))]
ax.barh(cmp_df['model'], cmp_df['mcc_mean'],
        xerr=cmp_df['mcc_std'], capsize=4,
        color=colors, edgecolor='black', linewidth=0.5)

ax.set_xlabel('Validation MCC (mean +/- std, 4 folds)')
ax.set_title('Model Comparison  -  Validation MCC', fontsize=13)
ax.axvline(0, color='black', linewidth=0.8)

best_model = cmp_df.iloc[0]['model']
ax.text(0.01, 0.97, f'Best: {best_model}', transform=ax.transAxes,
        va='top', fontsize=10, color='goldenrod', fontweight='bold')

plt.tight_layout()
plt.show()

### Multi-metric radar (spider) chart

In [ ]:
top_models = cmp_df.head(5)
metrics_radar = ['mcc_mean', 'f1_mean', 'bal_acc_mean', 'pr_auc_mean']
metric_labels = ['MCC', 'F1', 'Bal. Acc.', 'PR-AUC']

fig, axes = plt.subplots(1, len(top_models), figsize=(15, 3))
if len(top_models) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, top_models.iterrows()):
    vals = [row.get(m) or 0 for m in metrics_radar]
    ax.bar(metric_labels, vals, color=['steelblue', 'mediumseagreen', 'mediumpurple', 'darkorange'],
           edgecolor='black', linewidth=0.5)
    ax.set_ylim(0, 0.8)
    ax.set_title(f"{row['model']}\n{row.get('variant', '')}", fontsize=9)
    ax.tick_params(labelsize=8)

fig.suptitle('Top Models - Key Metrics', fontsize=12, y=1.05)
plt.tight_layout()
plt.show()


---
## 6  -  Reading real results from artifacts

After running `run_experiment.py`, JSON reports land in `artifacts/metrics/`. The cell below loads them if they exist.

In [ ]:
import json

metrics_dir = ROOT / 'artifacts' / 'metrics'
report_files = sorted(metrics_dir.glob('*_report.json'))

if not report_files:
    print('No artifact reports found yet. Run run_experiment.py first to generate real results.')
else:
    for jf in report_files:
        with open(jf, encoding='utf-8') as f:
            report = json.load(f)
        print(f'\n=== {jf.name} ===')
        best = report.get('best_cv_selection', {})
        final_test = report.get('final_test', {})
        metrics = final_test.get('metrics', {})
        print(f"  Best model : {best.get('model_name')} / {best.get('scaler_name')} / {best.get('variant_tag')}")
        print(f"  CV MCC     : {best.get('cv_summary', {}).get('mcc_mean')}")
        print(f"  Test MCC   : {metrics.get('mcc')}")
        print(f"  Test F1    : {metrics.get('f1')}")
        print(f"  Test PR-AUC: {metrics.get('pr_auc')}")
        print(f"  Selected probability threshold: {final_test.get('selected_probability_threshold')}")
        if final_test.get('baselines'):
            print('  Baselines:', ', '.join(final_test['baselines'].keys()))


---
## Summary

| Metric | Why we use it |
|---|---|
| MCC | Primary  -  single number, handles imbalance, accounts for all 4 confusion cells |
| F1 | Secondary  -  precision-recall trade-off |
| Balanced Accuracy | Secondary  -  mean per-class recall |
| ROC-AUC | Threshold-independent discriminative power |

The **best model is selected by `mcc_mean` over 4 CV folds**, then re-trained on the full development set and evaluated once on the locked test set.

**Next:** `08_full_pipeline.ipynb`  -  end-to-end orchestration via `run_experiment.py`.